# 実践プロジェクト：Netflix映画の俳優評価データ整理

## 分析目標

このデータ分析の目的は、コメディ、アクション、SFなど、異なるジャンルの映像作品について、各俳優が出演した作品の平均IMDb評価を整理し、各ジャンルにおける高評価作品の出演俳優を明らかにすること。

この実践プロジェクトの目的は、データ整理を練習し、次の分析に使えるデータを作成すること。

## プロジェクト概要

元データセットには、2022年7月時点でアメリカ地域から視聴可能なNetflixのテレビドラマおよび映画データが記録されている。データセットには、`titles.csv` と `credits.csv` の2つのデータ表が含まれる。

`titles.csv` には映画およびテレビドラマに関する情報が含まれており、映像作品ID、タイトル、種類、説明、ジャンル、IMDb（海外のオンライン評価サイト）評価などが含まれる。

`credits.csv` には、Netflix映像作品に出演した7万人以上の監督および俳優に関する情報が含まれており、名前、映像作品ID、人物名、出演者タイプ（監督／俳優）などが含まれる。

`titles.csv` 各列の意味は以下のとおり：

- `id`：映像作品ID。
- `title`：映像作品のタイトル。
- `show_type`：作品タイプ。テレビ番組または映画。
- `description`：簡単な説明。
- `release_year`：公開年。
- `age_certification`：年齢制限・レーティング。
- `runtime`：各テレビドラマのエピソードまたは映画の長さ。
- `genres`：ジャンルのリスト。
- `production_countries`：制作国のリスト。
- `seasons`：テレビドラマの場合、シーズン数。
- `imdb_id`：IMDbのID。
- `imdb_score`：IMDbの評価。
- `imdb_votes`：IMDbの投票数。
- `tmdb_popularity`：TMDBの人気度。
- `tmdb_score`：TMDBの評価。

`credits.csv` 各列の意味は以下のとおり：

- `person_ID`：出演者ID。
- `id`：参加した映像作品ID。
- `name`：名前。
- `character_name`：役名。
- `role`：出演者タイプ。俳優または監督。

## データの読み込み

In [171]:
import pandas as pd
original_titles = pd.read_csv("titles.csv")
original_credits = pd.read_csv("credits.csv")

## データの評価とクリーニング

クリーンアップ済みのデータと元のデータを区別するために、新しい変数 `cleaned_titles` を作成し、`original_titles` からコピーしたものを代入します。また、新しい変数 `cleaned_credits` を作成し、`original_credits` からコピーしたものを代入する。

以降のクリーニング処理は、すべて `cleaned_titles` と `cleaned_credits` に対して適用する。

In [172]:
cleaned_titles = original_titles.copy()
cleaned_credits = original_credits.copy()

### データの整然性

In [173]:
cleaned_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
5,ts22164,Monty Python's Flying Circus,SHOW,A British sketch comedy series with the shows ...,1969,TV-14,30,"['comedy', 'european']",['GB'],4.0,tt0063929,8.8,73424.0,17.617,8.306
6,tm70993,Life of Brian,MOVIE,"Brian Cohen is an average young Jewish man, bu...",1979,R,94,['comedy'],['GB'],NaN,tt0079470,8.0,395024.0,17.770,7.800
7,tm14873,Dirty Harry,MOVIE,When a madman dubbed 'Scorpio' terrorizes San ...,1971,R,102,"['thriller', 'action', 'crime']",['US'],NaN,tt0066999,7.7,155051.0,12.817,7.500
8,tm119281,Bonnie and Clyde,MOVIE,"In the 1930s, bored waitress Bonnie Parker fal...",1967,R,110,"['crime', 'drama', 'action']",['US'],NaN,tt0061418,7.7,112048.0,15.687,7.500
9,tm98978,The Blue Lagoon,MOVIE,Two small children and a ship's cook survive a...,1980,R,104,"['romance', 'action', 'drama']",['US'],NaN,tt0080453,5.8,69844.0,50.324,6.156


データの一部である10行を見ると、`cleaned_titles` の `genres` と `production_countries` の変数には複数の値が含まれているため、分割する必要がある。

まず、任意の `genres` 変数の値を1つ抽出して確認する。

In [174]:
cleaned_titles["genres"][10]

"['action', 'drama', 'war']"

`genres` はリストのような形式で表示されているが、実際の型は文字列のリストではなく文字列である。そのため、`value_counts` を使って各値の出現回数を直接集計することはできない。

Python 組み込みの `eval` 関数を使うことで、文字列を式として変換できる。これにより、リストを表す文字列をリストそのものに変換できる。

In [175]:
cleaned_titles["genres"] = cleaned_titles["genres"].apply(eval)
cleaned_titles["genres"][10]

['action', 'drama', 'war']

リストに変換した後は、DataFrame の `explode` メソッドを使って、その列のリスト値を個別の行に分割できる。

In [176]:
cleaned_titles = cleaned_titles.explode(["genres"])
cleaned_titles.sample(5)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
4037,tm975398,Christmas Crossfire,MOVIE,"A man foils an attempted murder, then flees th...",2020,NaN,106,crime,['DE'],NaN,tt13446424,4.8,1373.0,6.882,4.5
2179,tm420656,Reprisal,MOVIE,"Jacob, a bank manager haunted by a violent hei...",2018,R,89,crime,"['GB', 'US']",NaN,tt6547170,4.2,7504.0,13.316,5.3
3407,tm452851,The Operative,MOVIE,A young Western woman is recruited by the Moss...,2019,NaN,120,action,"['IL', 'US', 'FR', 'DE']",NaN,tt8000718,5.7,6250.0,18.028,5.7
4802,ts251675,Hellbound,SHOW,"Unearthly beings deliver bloody condemnations,...",2021,TV-MA,52,crime,['KR'],1.0,tt12235718,6.6,23099.0,193.725,7.6
3259,tm845496,Crip Camp: A Disability Revolution,MOVIE,Down the road from Woodstock in the early 1970...,2020,R,107,history,['US'],NaN,tt8923484,7.7,7428.0,10.547,7.3


同様に、各観測値の production_countries の値は単一のジャンルを表すものではなく、一連のジャンルを表している。

まず、任意の production_countries 変数の値を1つ抽出して確認する。

In [177]:
cleaned_titles["production_countries"][10]

10    ['GB', 'US']
10    ['GB', 'US']
10    ['GB', 'US']
Name: production_countries, dtype: str

`production_countries` も同じ問題がある。リストのような形式で表示されているが、実際の型は文字列のリストではなく文字列であるため、分割しにくい。

再び `eval` 関数を利用して型変換を行い、変換後に確かにリスト型になっていることを確認する。

In [178]:
cleaned_titles["production_countries"] = cleaned_titles["production_countries"].apply(eval)
cleaned_titles["production_countries"][10]

10    [GB, US]
10    [GB, US]
10    [GB, US]
Name: production_countries, dtype: object

In [179]:
cleaned_titles = cleaned_titles.explode("production_countries")
cleaned_titles["production_countries"][10]

10    GB
10    US
10    GB
10    US
10    GB
10    US
Name: production_countries, dtype: str

`cleaned_titles` の構造的な問題を処理した後、`cleaned_credits` を確認する。

In [180]:
cleaned_credits.sample(10)

,person_id,id,name,character,role
35107,1329803,tm285295,John C. Ashton,Old Rudy,ACTOR
25954,397245,tm230524,Liliana Castro,Maria Eudóxia's sister,ACTOR
67603,18347,tm861000,Alfredo Narciso,Defense Lawyer / William,ACTOR
26832,262826,tm224275,Hana Chamoun,Fidaa,ACTOR
6327,1831391,tm35463,Jodie Mann,Nurse (Cape Cod),ACTOR
44421,82885,tm244149,Gino Cafarelli,Mayor Frank Rizzo,ACTOR
43646,719863,tm452767,Claire Ashton,Beach Goer,ACTOR
23738,12025,tm233616,Roger Craig Smith,Captain America,ACTOR
46515,866848,tm843444,Jordan Morgan,Dennis The Menace (Radio Man),ACTOR
24189,97310,ts42038,Amy Pietz,Deirdre,ACTOR


ランダムに抽出した10行のデータを見ると、`cleaned_credits` のデータは「各変数が1列、各観測値が1行、各種類の観測単位が1つの表」という形式に従っている。そのため、構造的な問題は存在しない。

### データの清潔度

`info` を通じて、データ内容を大まかに把握する。

In [181]:
cleaned_titles.info()

<class 'pandas.DataFrame'>
Index: 17818 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    17818 non-null  str    
 1   title                 17817 non-null  str    
 2   type                  17818 non-null  str    
 3   description           17790 non-null  str    
 4   release_year          17818 non-null  int64  
 5   age_certification     10889 non-null  str    
 6   runtime               17818 non-null  int64  
 7   genres                17755 non-null  str    
 8   production_countries  17439 non-null  str    
 9   seasons               6224 non-null   float64
 10  imdb_id               17116 non-null  str    
 11  imdb_score            16976 non-null  float64
 12  imdb_votes            16945 non-null  float64
 13  tmdb_popularity       17663 non-null  float64
 14  tmdb_score            17241 non-null  float64
dtypes: float64(5), int64(2), str(8)
memo

出力結果を見ると、`cleaned_titles` データには合計17818件の観測値がある。`title`、`description`、`age_certification`、`genres`、`production_countries`、`seasons`、`imdb_id`、`imdb_score`、`tmdb_popularity`、`tmdb_score`、`imdb_votes`、`tmdb_popularity`、`tmdb_score` の各変数には欠損値が存在するため、後続で評価とクリーニングを行う。

また、`release_year` は年を表しているため、データ型は数値ではなく日付型であるべきである。そのため、データ形式の変換が必要である。

In [182]:
cleaned_titles["release_year"] = pd.to_datetime(cleaned_titles["release_year"], format="%Y")
cleaned_titles["release_year"]

0      1945-01-01
1      1976-01-01
1      1976-01-01
2      1972-01-01
2      1972-01-01
          ...    
5847   2021-01-01
5848   2021-01-01
5849   2021-01-01
5849   2021-01-01
5849   2021-01-01
Name: release_year, Length: 17818, dtype: datetime64[us]

In [183]:
cleaned_credits.info()

<class 'pandas.DataFrame'>
RangeIndex: 77801 entries, 0 to 77800
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   person_id  77801 non-null  int64
 1   id         77801 non-null  str  
 2   name       77801 non-null  str  
 3   character  68029 non-null  str  
 4   role       77801 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.0 MB


出力結果を見ると、`cleaned_credits` データには合計77801件の観測値がある。そのうち、`character` 変数には欠損値が存在するため、後続で評価とクリーニングを行う。

また、`person_id` はキャスト・スタッフのIDを表しているため、データ型は数値ではなく文字列であるべきである。そのため、データ形式の変換が必要である。

In [184]:
cleaned_credits["person_id"] = cleaned_credits["person_id"].astype(str)
cleaned_credits["person_id"]

0           3748
1          14658
2           7064
3           3739
4          48933
          ...   
77796     736339
77797     399499
77798     373198
77799     378132
77800    1950416
Name: person_id, Length: 77801, dtype: str

#### 欠損データの処理

`cleaned_titles` では、`title`、`description`、`age_certification`、`genres`、`production_countries`、`seasons`、`imdb_id`、`imdb_score`、`tmdb_popularity`、`tmdb_score`、`imdb_votes`、`tmdb_popularity`、`tmdb_score` の各変数に欠損値が存在する。

映画・ドラマ作品のタイトル、説明、年齢認証、制作国、ドラマのシーズン数、IMDB の ID、TMDB の人気度、TMDB の評価は、各ジャンルにおける高 IMDB 評価作品の俳優を抽出する分析には影響しない。そのため、`title`、`description`、`age_certification`、`production_countries`、`seasons`、`imdb_id`、`tmdb_popularity`、`tmdb_score`、`imdb_votes`、`tmdb_popularity`、`tmdb_score` の変数値に欠損がある観測値は残しておくことができる。

一方で、`imdb_score` と `genres`、つまり IMDB 評価とジャンルは、後続の分析と密接に関係している。

まず、`imdb_score` が欠損している観測値を抽出して確認する。

In [185]:
cleaned_titles.query("imdb_score.isnull()")

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945-01-01,TV-MA,51,documentation,US,1.0,NaN,NaN,NaN,0.600,NaN
75,tm132164,Bill Hicks: Sane Man,MOVIE,Sane Man was filmed before Bill recorded ‘Dang...,1989-01-01,R,80,comedy,US,NaN,NaN,NaN,NaN,3.377,7.5
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,documentation,JP,12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,family,JP,12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,reality,JP,12.0,NaN,NaN,NaN,7.730,7.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5810,tm1225897,Social Man,MOVIE,Two competitive social media Influencers go he...,2021-01-01,NaN,96,drama,NaN,NaN,tt20198164,NaN,NaN,NaN,NaN
5833,ts307884,HQ Barbers,SHOW,When a family run barber shop in the heart of ...,2021-01-01,TV-14,24,comedy,NG,1.0,NaN,NaN,NaN,0.840,NaN
5840,tm1216735,Sun of the Soil,MOVIE,"In 14th-century Mali, an ambitious young royal...",2022-01-01,NaN,26,NaN,NaN,NaN,NaN,NaN,NaN,1.179,7.0
5844,tm1074617,Bling Empire - The Afterparty,MOVIE,"The stars of ""Bling Empire"" discuss the show's...",2021-01-01,NaN,35,NaN,US,NaN,NaN,NaN,NaN,NaN,NaN


分析に必要な核心データである `imdb_score` が欠損しているため、これらの観測値を削除し、削除後にこの列の欠損値の個数を確認する。

In [186]:
cleaned_titles = cleaned_titles.dropna(subset=["imdb_score"])
cleaned_titles["imdb_score"].isnull().sum()

np.int64(0)

次に、`genres` が欠損している観測値を抽出して確認する。

In [187]:
cleaned_titles.query("genres.isnull()")

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1813,ts77824,My Next Guest Needs No Introduction With David...,SHOW,TV legend David Letterman teams up with fascin...,2018-01-01,TV-MA,50,NaN,US,4.0,tt7829834,7.8,5581.0,8.217,7.6
1939,ts215037,Minecraft: Story Mode,SHOW,"MInecraft: Story Mode is an interactive, anima...",2018-01-01,TV-PG,52,NaN,US,1.0,tt10498322,5.6,347.0,NaN,NaN
2386,ts74805,A Little Help with Carol Burnett,SHOW,In this unscripted series starring comedy lege...,2018-01-01,TV-G,24,NaN,US,1.0,tt7204366,6.3,237.0,1.621,6.2
2658,ts265844,#ABtalks,SHOW,#ABtalks is a YouTube interview show hosted by...,2018-01-01,TV-PG,68,NaN,NaN,1.0,tt12635254,9.6,7.0,NaN,NaN
4274,tm1172010,The Lockdown Plan,MOVIE,NaN,2020-01-01,NaN,49,NaN,NaN,NaN,tt13079112,6.5,NaN,NaN,NaN
4648,tm1113921,In Vitro,MOVIE,'In Vitro' is an otherworldly rumination on me...,2019-01-01,NaN,27,NaN,NaN,NaN,tt10545994,7.7,NaN,NaN,NaN


分析に必要な核心データである `genres` が欠損しているため、これらの観測値を削除し、削除後にこの列の欠損値の個数を確認する。

In [188]:
cleaned_titles = cleaned_titles.dropna(subset=["genres"])
cleaned_titles["genres"].isnull().sum()

np.int64(0)

次に、`cleaned_credits` の欠損データを評価する。このデータでは、`character` 変数のみに欠損値が存在する。

役名は、各ジャンルにおける高 IMDB 評価作品の俳優を抽出する分析には影響しない。また、この変数の欠損は、演職員の区分が監督であり、対応する役名が存在しないことが原因である可能性もある。そのため、`character` 変数値に欠損がある観測値は残しておくことができる。

### 重複データの処理

データ変数の意味と内容から見ると、`cleaned_titles` にはすべての変数値が同じ観測値が存在すべきではない。そのため、重複値が存在するかどうかを確認する。

In [189]:
cleaned_titles.duplicated().sum()

np.int64(0)

In [190]:
cleaned_credits.duplicated().sum()

np.int64(0)

### 不一致データの処理

`cleaned_titles` について、不一致データは `genres` と `character` 変数に存在する可能性がある。複数の異なる値が同一のジャンルを指している場合や、複数の異なる値が同一の国を指している場合があるかどうかを確認する。

In [191]:
cleaned_titles["genres"].value_counts()

genres
drama            3357
comedy           2419
thriller         1446
action           1339
romance          1080
crime            1066
documentation     981
family            769
animation         732
fantasy           727
european          679
scifi             647
horror            438
history           336
music             266
reality           226
war               221
sport             188
western            53
Name: count, dtype: int64

In [192]:
cleaned_titles['production_countries'].value_counts()

production_countries
US    5648
IN    1610
GB    1068
JP    1046
FR     720
      ... 
NP       1
LK       1
GT       1
AF       1
FO       1
Name: count, Length: 108, dtype: int64

`value_counts` の実行結果には値が多すぎるため、Pandas はデフォルトで先頭と末尾の一部の値だけを表示する。

結果を完全に表示するには、`display.max_rows` を `None` に設定し、表示行数の上限を解除できる。

ただし、今回は現在の `value_counts` の呼び出し時だけ完全な結果を確認したい。そのため、`option_context` と組み合わせて、一時的に表示上限だけを変更する。

In [193]:
with pd.option_context('display.max_rows', None):
    print(cleaned_titles['production_countries'].value_counts())

production_countries
US         5648
IN         1610
GB         1068
JP         1046
FR          720
ES          637
KR          637
CA          608
DE          383
CN          295
MX          264
IT          224
BR          221
AU          217
TR          195
PH          192
AR          150
ID          149
BE          148
TW          133
NG          131
PL          126
ZA          103
HK          102
NL          102
CO           94
EG           93
DK           89
TH           87
SE           81
LB           70
NO           68
AE           52
IE           49
SG           47
XX           43
IL           42
RU           41
CL           35
CH           33
PS           32
BG           31
MY           30
AT           28
SA           28
IS           28
NZ           27
LU           27
PE           26
RO           25
QA           24
CZ           22
JO           19
HU           18
FI           18
MA           15
UY           15
PT           14
KW           10
KH           10
PK            9
PR 

以上の出力結果を見ると、制作国はすべて2文字の国コードで表されているが、その中に `Lebanon` という値が1つ存在している。

`Lebanon` の国コードは `LB` であり、`LB` は39回出現している。つまり、この部分にはデータの不一致がある。`LB` と `Lebanon` は同じ国を表しているため、表記を統一する必要がある。

`cleaned_titles` の `production_countries` における `"LB"` と `"Lebanon"` を `LB` に統一し、置換後に `"LB"` がまだ存在するかどうかを確認する。

In [194]:
cleaned_titles['production_countries'] = cleaned_titles['production_countries'].replace("Lebanon", "LB")
with pd.option_context('display.max_rows', None):
    print(cleaned_titles['production_countries'].value_counts())

production_countries
US    5648
IN    1610
GB    1068
JP    1046
FR     720
ES     637
KR     637
CA     608
DE     383
CN     295
MX     264
IT     224
BR     221
AU     217
TR     195
PH     192
AR     150
ID     149
BE     148
TW     133
NG     131
PL     126
ZA     103
HK     102
NL     102
CO      94
EG      93
DK      89
TH      87
SE      81
LB      71
NO      68
AE      52
IE      49
SG      47
XX      43
IL      42
RU      41
CL      35
CH      33
PS      32
BG      31
MY      30
AT      28
SA      28
IS      28
NZ      27
LU      27
PE      26
RO      25
QA      24
CZ      22
JO      19
HU      18
FI      18
MA      15
UY      15
PT      14
KW      10
KH      10
PK       9
PR       9
MT       8
UA       8
VN       8
SU       7
CD       7
IR       7
TN       7
LT       7
GH       6
KE       6
AL       6
SN       6
IQ       5
CY       5
MU       5
SY       4
TZ       4
MC       4
IO       4
GR       4
KN       4
DZ       3
BS       3
HR       3
GL       3
PY       3
CM       3


また、空文字列で表された国コードも存在しており、有効なデータではない。

ただし、制作国は分析に必要な重要情報ではないため、制作国が空の観測値は残しておくことができる。

`original_credits` について、不一致データは `role` に存在する可能性がある。複数の異なる値が同一の演職員タイプを指している場合があるかどうかを確認する。

In [195]:
original_credits["role"].value_counts()

role
ACTOR       73251
DIRECTOR     4550
Name: count, dtype: int64

以上の出力結果を見ると、`role` には `ACTOR` または `DIRECTOR` の2種類の値しか存在せず、不一致データは存在しない。

この列の型を `Category` に変換できる。文字列型よりもメモリ使用量を節約でき、値の種類が限定されていることも示せる。

In [196]:
cleaned_credits["role"] = cleaned_credits["role"].astype("category")
cleaned_credits["role"]

0           ACTOR
1           ACTOR
2           ACTOR
3           ACTOR
4           ACTOR
           ...   
77796       ACTOR
77797       ACTOR
77798       ACTOR
77799       ACTOR
77800    DIRECTOR
Name: role, Length: 77801, dtype: category
Categories (2, str): ['ACTOR', 'DIRECTOR']

### 無効データまたは誤ったデータの処理

In [197]:
cleaned_titles.describe()

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
count,16970,16970.000000,5954.000000,16970.000000,1.694100e+04,16842.000000,16515.000000
mean,2015-11-14 22:42:51.974072,80.912552,2.455492,6.514207,3.281655e+04,29.396307,6.846933
min,1954-01-01 00:00:00,0.000000,1.000000,1.500000,5.000000e+00,0.600000,1.000000
25%,2015-01-01 00:00:00,45.000000,1.000000,5.800000,7.800000e+02,4.070000,6.200000
50%,2018-01-01 00:00:00,90.000000,2.000000,6.600000,3.508000e+03,10.195000,6.900000
75%,2020-01-01 00:00:00,107.000000,3.000000,7.300000,1.697800e+04,23.639000,7.500000
max,2022-01-01 00:00:00,225.000000,42.000000,9.500000,2.294231e+06,2274.044000,10.000000
std,NaN,39.596172,2.869428,1.131095,1.141492e+05,93.178235,1.078831


In [198]:
cleaned_credits.describe()

,person_id,id,name,character,role
count,77801,77801,77801,68029,77801
unique,54589,5489,54314,47274,2
top,48004,tm32982,Kareena Kapoor Khan,Self,ACTOR
freq,25,208,25,1950,73251


以上の統計情報を見ると、現実的な意味から外れた数値は存在しない。

## データの整理

今回のデータ分析の目的は、コメディ、アクション、SF など、異なるジャンルの映画・ドラマ作品において、俳優が出演した作品の平均 IMDB 評価を整理し、各ジャンルにおける高評価作品の俳優を抽出することである。

ジャンルデータと俳優データを同時に取得するために、`id` をキーとして `cleaned_titles` と `cleaned_credits` を結合する必要がある。

In [199]:
credits_with_titles = pd.merge(original_credits, cleaned_titles, on="id", how="inner")

In [200]:
credits_with_titles.sample(5)

,person_id,id,name,character,role,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
83596,66333,ts38090,Evan Williams,The Knight of Lorraine,ACTOR,Versailles,SHOW,The story of a young Louis XIV on his journey ...,2015-01-01,TV-MA,54,documentation,FR,3.0,tt3830558,7.9,16505.0,31.857,7.800
131094,42566,tm351101,Ramon Agirre,Alfredo,ACTOR,Errementari: The Blacksmith and the Devil,MOVIE,"Basque Country, Spain, 1843. A police constabl...",2018-01-01,NaN,98,horror,FR,NaN,tt5592878,6.4,11653.0,15.977,6.377
156397,5646,ts85250,Jess Harnell,NaN,ACTOR,Kulipari: Dream Walker,SHOW,"Now the Blue Sky King, Darel must lead a rescu...",2018-01-01,TV-Y7,23,action,US,1.0,tt9203064,7.6,92.0,3.908,9.000
39620,74584,tm110923,Elia Suleiman,E.S.,ACTOR,Divine Intervention,MOVIE,Santa Claus tries to outrun a gang of knife-wi...,2002-01-01,NaN,92,war,DE,NaN,tt0274428,6.6,3598.0,2.894,6.400
195153,1052122,tm465724,Boma Akpore,Funeral attendee (uncredited),ACTOR,See You Yesterday,MOVIE,As two teen prodigies try to master the art of...,2019-01-01,NaN,86,thriller,US,NaN,tt8743064,5.2,10725.0,7.583,5.600


俳優の出演作品の評価のみを抽出するため、監督は分析対象外である。

そのため、`role` に基づいて、種類が `ACTOR` である観測値を抽出し、後続の分析に使用する。

In [201]:
actor_with_titles = credits_with_titles.query("role == 'ACTOR'")

各ジャンルにおける高 IMDB 評価作品の俳優を抽出するために、まずジャンルと俳優に基づいてグループ化する必要がある。

俳優でグループ化する際は、`person_id` 変数を使用する。

In [202]:
groupby_genres_and_person_id = actor_with_titles.groupby(["genres", "person_id"])

グループ化した後は、`imdb_score` の値に対して集計計算を行う必要がある。

そのため、`imdb_score` 変数のみを抽出し、`mean` を呼び出して、各ジャンルの映画・ドラマ作品における各俳優の出演作品の平均 IMDB 評価を計算する。

In [203]:
imdb_score_groupby_genres_and_person_id = groupby_genres_and_person_id["imdb_score"].mean()
imdb_score_groupby_genres_and_person_id

genres   person_id
action   45           5.0
         48           5.4
         51           6.4
         53           6.8
         54           5.3
                     ... 
western  2353339      6.9
         2370848      6.1
         2398539      3.8
         2406218      6.0
         2408082      7.3
Name: imdb_score, Length: 168881, dtype: float64

`reset_index` を呼び出して階層化インデックスをリセットし、より整った DataFrame を得る。

In [204]:
imdb_score_groupby_genres_and_person_id_df = imdb_score_groupby_genres_and_person_id.reset_index()
imdb_score_groupby_genres_and_person_id_df

,genres,person_id,imdb_score
0,action,45,5.0
1,action,48,5.4
2,action,51,6.4
3,action,53,6.8
4,action,54,5.3
...,...,...,...
168876,western,2353339,6.9
168877,western,2370848,6.1
168878,western,2398539,3.8
168879,western,2406218,6.0


上の結果に対して再度グループ化を行い、各ジャンルにおいて俳優の出演作品の平均評価の最高値はいくつか、その最高評価に対応する俳優名は誰かを抽出する。

この結果を得るには、再び `genres` でグループ化し、`imdb_score` 変数を抽出して、その最大値を計算する必要がある。

In [205]:
genres_max_scores = imdb_score_groupby_genres_and_person_id_df.groupby("genres")["imdb_score"].max()
genres_max_scores

genres
action           9.3
animation        9.3
comedy           9.2
crime            9.5
documentation    9.1
drama            9.5
european         8.9
family           9.3
fantasy          9.3
history          9.1
horror           9.0
music            8.8
reality          8.9
romance          9.2
scifi            9.3
sport            9.1
thriller         9.5
war              8.8
western          8.9
Name: imdb_score, dtype: float64

最高点が分かった後は、上記の結果と、以前得られた `imdb_score_groupby_genres_and_person_id_df` を再度結合する。

これにより、最高点に対応する各俳優 ID が何であるかを取得できる。

In [206]:
genres_max_score_with_person_id = pd.merge(imdb_score_groupby_genres_and_person_id_df, genres_max_scores, on=["genres", "imdb_score"])
genres_max_score_with_person_id

,genres,person_id,imdb_score
0,action,1303,9.3
1,action,12790,9.3
2,action,21033,9.3
3,action,86591,9.3
4,action,336830,9.3
...,...,...,...
131,war,826547,8.8
132,western,22311,8.9
133,western,28166,8.9
134,western,28180,8.9


以上の結果から、最高点に対応する俳優は必ずしも1人だけではなく、複数の俳優が同じ平均評価を持つ可能性がある。

俳優 ID に対応する俳優名を取得するために、`cleaned_credits` という DataFrame と結合できる。この DataFrame には他の列も含まれているが、必要なのは `person_id` と `name` の対応関係だけである。そのため、まずこの2列だけを抽出し、重複行を削除する。

In [207]:
actor_id_with_names = cleaned_credits[['person_id', 'name']].drop_duplicates()
actor_id_with_names.head(10)

,person_id,name
0,3748,Robert De Niro
1,14658,Jodie Foster
2,7064,Albert Brooks
3,3739,Harvey Keitel
4,48933,Cybill Shepherd
5,32267,Peter Boyle
6,519612,Leonard Harris
7,29068,Diahnne Abbott
8,519613,Gino Ardito
9,3308,Martin Scorsese


次のステップでは、actor_id_with_names と前に得られた genres_max_score_with_person_id を結合し、name 変数を追加する。

これにより、平均評価が最も高い俳優名を表示できる。

In [208]:
genres_max_score_with_person_id["person_id"] = genres_max_score_with_person_id["person_id"].astype(str)
genres_max_score_with_actor_name = pd.merge(genres_max_score_with_person_id, actor_id_with_names, on="person_id")
genres_max_score_with_actor_name

,genres,person_id,imdb_score,name
0,action,1303,9.3,Jessie Flower
1,action,12790,9.3,Olivia Hack
2,action,21033,9.3,Zach Tyler
3,action,86591,9.3,Cricket Leigh
4,action,336830,9.3,André Sogliuzzo
...,...,...,...,...
131,war,826547,8.8,Yuto Uemura
132,western,22311,8.9,Koichi Yamadera
133,western,28166,8.9,Megumi Hayashibara
134,western,28180,8.9,Unsho Ishizuka


同じジャンルをまとめて並べるために、`sort_values` メソッドを使い、結果の行を `genres` に基づいて並べ替える。その後、`reset_index` を使ってインデックスを振り直す。

インデックスを振り直すと、DataFrame に `index` 列が追加されるため、さらに `index` 列を削除する。

In [209]:
genres_max_score_with_actor_name = genres_max_score_with_actor_name.sort_values("genres").reset_index().drop("index", axis=1)
genres_max_score_with_actor_name

,genres,person_id,imdb_score,name
0,action,1303,9.3,Jessie Flower
1,action,12790,9.3,Olivia Hack
2,action,21033,9.3,Zach Tyler
3,action,86591,9.3,Cricket Leigh
4,action,336830,9.3,André Sogliuzzo
...,...,...,...,...
131,war,826547,8.8,Yuto Uemura
132,western,28180,8.9,Unsho Ishizuka
133,western,22311,8.9,Koichi Yamadera
134,western,28166,8.9,Megumi Hayashibara
